<a href="https://colab.research.google.com/github/wendysbustillo-sketch/SIMULACION-DE-SISTEMAS/blob/main/Practica4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install simpy
import simpy
import random

# Parámetros del sistema
TIEMPO_MOLDEO = 30        # segundos (media)
TIEMPO_ENFRIAMIENTO = 60  # segundos (constante)
TIEMPO_INSPECCION = 45    # segundos (media)
TAMANO_LOTE_EMPAQUE = 100 # botellas por lote
NUM_MAQUINAS_MOLDEO = 2
NUM_INSPECTORES = 1
PROB_DEFECTO = 0.15        # probabilidad de que una botella sea defectuosa

# Proceso de producción de una botella
def producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas):
    # Moldeo
    with moldeo.request() as req:
        yield req
        tiempo = random.normalvariate(TIEMPO_MOLDEO, 5)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} moldeada en {env.now:.1f}")

    # Enfriamiento
    yield env.timeout(TIEMPO_ENFRIAMIENTO)
    print(f"Botella {id_botella} enfriada en {env.now:.1f}")

    # Inspección
    with inspeccion.request() as req:
        yield req
        tiempo = random.expovariate(1.0/TIEMPO_INSPECCION)
        yield env.timeout(tiempo)

        # Decisión: ¿aprobada o defectuosa?
        if random.random() < PROB_DEFECTO:
            descartadas.append(id_botella)
            print(f"❌ Botella {id_botella} descartada en {env.now:.1f}")
            return
        else:
            print(f"✅ Botella {id_botella} aprobada en {env.now:.1f}")

    # Empaque (se acumulan hasta formar un lote)
    empaque.append(id_botella)
    if len(empaque) >= TAMANO_LOTE_EMPAQUE:
        print(f"📦 Lote de {TAMANO_LOTE_EMPAQUE} botellas empacado en {env.now:.1f}")
        empaque.clear()

# Generador de llegadas de botellas
def llegada_botellas(env, moldeo, inspeccion, empaque, descartadas):
    id_botella = 0
    while True:
        yield env.timeout(random.expovariate(1/20))  # llegada cada ~20 seg
        id_botella += 1
        env.process(producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas))

# Configuración del entorno
env = simpy.Environment()
moldeo = simpy.Resource(env, NUM_MAQUINAS_MOLDEO)
inspeccion = simpy.Resource(env, NUM_INSPECTORES)
empaque = []
descartadas = []

# Iniciar simulación
env.process(llegada_botellas(env, moldeo, inspeccion, empaque, descartadas))
env.run(until=500)  # simular 500 segundos

# Resumen final
print("\n--- RESULTADOS ---")
print(f"Botellas descartadas: {len(descartadas)}")
print(f"IDs descartados: {descartadas}")

Botella 1 moldeada en 58.3
Botella 2 moldeada en 107.9
Botella 1 enfriada en 118.3
Botella 3 moldeada en 127.6
✅ Botella 1 aprobada en 143.1
Botella 4 moldeada en 144.3
Botella 2 enfriada en 167.9
✅ Botella 2 aprobada en 179.4
Botella 3 enfriada en 187.6
Botella 6 moldeada en 202.8
Botella 4 enfriada en 204.3
Botella 5 moldeada en 204.5
Botella 7 moldeada en 237.9
Botella 8 moldeada en 239.8
✅ Botella 3 aprobada en 241.4
❌ Botella 4 descartada en 249.2
Botella 6 enfriada en 262.8
Botella 5 enfriada en 264.5
Botella 9 moldeada en 267.3
✅ Botella 6 aprobada en 269.2
Botella 10 moldeada en 269.5
Botella 11 moldeada en 295.1
Botella 12 moldeada en 297.0
Botella 7 enfriada en 297.9
Botella 8 enfriada en 299.8
✅ Botella 5 aprobada en 312.1
❌ Botella 7 descartada en 315.8
Botella 14 moldeada en 316.5
Botella 13 moldeada en 325.0
Botella 9 enfriada en 327.3
Botella 10 enfriada en 329.5
✅ Botella 8 aprobada en 347.5
Botella 15 moldeada en 349.3
Botella 11 enfriada en 355.1
Botella 12 enfriada e